In [7]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
            {
                "ToxPipeMCPServers": {
                    "transport": "http",
                    "url": "http://127.0.0.1:9223/mcp",
                }
            }
        )

tools = await client.get_tools()


In [ ]:
import importlib
import json
import pandas as pd
utils = importlib.import_module("eval-app.src.utils")

d_providers = {
    'o3': 'o3',
    'Toxpipe (RAG) [o3]': 'o3',
    'GPT-4o': 'GPT-4o',
    'Claude 3.7 Sonnet': 'Claude 3.7 Sonnet',
    'Gemini 2.5 Pro': 'Gemini 2.5 Pro',
    'Gemini 2.5 Flash': 'Gemini 2.5 Flash',
    'Llama 3.3 70B': 'Llama 3.3 70B',
    'Mistral Large 2': 'Mistral Large 2',
    'Toxpipe (RAG) [GPT-4o]': 'GPT-4o',
    'Toxpipe (RAG) [Claude 3.7 Sonnet]': 'Claude 3.7 Sonnet',
    'Toxpipe (RAG) [Gemini 2.5 Pro]': 'Gemini 2.5 Pro',
    'Toxpipe (RAG) [Gemini 2.5 Flash]': 'Gemini 2.5 Flash',
    'Toxpipe (RAG) [Llama 3.3 70B]': 'Llama 3.3 70B',
    'Toxpipe (RAG) [Mistral Large 2]': 'Mistral Large 2',
    'Toxpipe (Agentic) [GPT-4o]': 'GPT-4o'
}

for prompt_type in ['abt-qa-assertion-prompts_mixed', 'tox-type-assertion-prompts_human', 'tox-type-assertion-prompts_rat']:

    df = pd.DataFrame(columns=['provider', 'prompt']).set_index(['provider', 'prompt'])

    for level in ['base-model', 'rag']:

        eval_name = f'{level}_{prompt_type}'

        df_eval = pd.DataFrame()

        for f in (utils.Config.DIR_TEST / eval_name / 'output').glob('output_*.json'):
            with open(f) as fp:
                d = json.load(fp)

            r = {}
            for t in d['tests']:
                level_accessed = 'base'
                if level == 'rag' and t['response']['steps_taken'][-1] == 'query_with_context':
                    level_accessed = 'rag'

                result =  'no-assertion' if not t['response']['results'] else 'pass' if t['response']['results']['pass'] else 'fail'

                r[f'provider'] = r.get(f'provider', []) + [d_providers[t['provider']['label']]]
                r[f'prompt'] = r.get(f'prompt', []) + [t['prompt'].format(**t['vars'])]
                r[f'level_{level}'] = r.get(f'level_{level}', []) + [level_accessed]
                r[f'result_{level}'] = r.get(f'result_{level}', []) + [result]

            df_eval = pd.concat([df_eval, pd.DataFrame(r)], axis=0).reset_index(drop=True)
    
        df = df.join(df_eval.set_index(['provider', 'prompt']), how='outer')

    rag_success = df[(df['result_base-model'] == 'fail') & (df['level_rag'] == 'rag') & (df['result_rag'] == 'pass')].reset_index()

    print(prompt_type)

    display(rag_success)
    print(rag_success['prompt'].values)

    rag_success['prompt']
    print(len(rag_success))

abt-qa-assertion-prompts_mixed


,provider,prompt,level_base-model,result_base-model,level_rag,result_rag
0,Claude 3.7 Sonnet,Consider the following question followed by op...,base,fail,rag,pass
1,Claude 3.7 Sonnet,Consider the following question followed by op...,base,fail,rag,pass
2,Claude 3.7 Sonnet,Consider the following question followed by op...,base,fail,rag,pass
3,Claude 3.7 Sonnet,Consider the following question followed by op...,base,fail,rag,pass
4,Claude 3.7 Sonnet,Consider the following question followed by op...,base,fail,rag,pass
...,...,...,...,...,...,...
82,o3,Consider the following question followed by op...,base,fail,rag,pass
83,o3,Consider the following question followed by op...,base,fail,rag,pass
84,o3,Consider the following question followed by op...,base,fail,rag,pass
85,o3,Consider the following question followed by op...,base,fail,rag,pass


['Consider the following question followed by options. Answer the question from choosing the options. You can choose multiple options. Each option must start with the option number.\n**Question**:\nA comparison of newborn rats and human infants shows that\n**Options**:\n(A) the blood brain barrier is more developed in the newborn rat than the infant\n\n(B) renal function\n\n(C) drug-metabolizing enzymes\n\n(D) neuronal synapses in the central nervous system are complete by birth in the neonatal infant and rat\n\n(E) newborn rats and human infants are developmentally similar at birth'
 'Consider the following question followed by options. Answer the question from choosing the options. You can choose multiple options. Each option must start with the option number.\n**Question**:\nA semi-conscious pesticide sprayer is brought into the emergency room from a nearby orchard with acute carbaryl poisoning that occurred within the last 45 minutes. He is starting to have trouble breathing. In ad

,provider,prompt,level_base-model,result_base-model,level_rag,result_rag
0,Claude 3.7 Sonnet,Answer the following question with a list of s...,base,fail,rag,pass
1,Llama 3.3 70B,Answer the following question with a list of s...,base,fail,rag,pass
2,Mistral Large 2,Answer the following question with a list of s...,base,fail,rag,pass
3,o3,Answer the following question with a list of s...,base,fail,rag,pass
4,o3,Answer the following question with a list of s...,base,fail,rag,pass


['Answer the following question with a list of string. List the Toxicokinetic parameters of Dioxin or its synonyms with CAS number 1746-01-6 on human'
 'Answer the following question with a list of string. List the Toxicokinetic parameters of Aspirin or its synonyms with CAS number 50-78-2 on human'
 'Answer the following question with a list of string. List the Toxicokinetic parameters of Aspirin or its synonyms with CAS number 50-78-2 on human'
 'Answer the following question with a list of string. List the Nervous system/Neurological toxicity of Methylmercury or its synonyms with CAS number 16056-34-1 on human'
 'Answer the following question with a list of string. List the Respiratory system toxicity of Paracetamol or its synonyms with CAS number 103-90-2 on human']
5
tox-type-assertion-prompts_rat


,provider,prompt,level_base-model,result_base-model,level_rag,result_rag


[]
0


In [ ]:
eval = importlib.import_module("eval-app.src.evaluation")
import json

output = json.loads("""\
           {
                "id": "202507-2513-3935-96e689ec-dbbc-4cad-a4c6-db69d8478896",
                "system_prompt": "You are an expert toxicologist with extensive knowledge in chemical safety assessment, toxicokinetics, and toxicodynamics. Your expertise includes:\n1. Interpreting chemical structures and properties 2. Analyzing toxicological data from various sources (e.g., in vitro, in vivo, and in silico studies) 3. Applying read-across and QSAR (Quantitative Structure-Activity Relationship) approaches 4. Understanding mechanisms of toxicity and adverse outcome pathways 5. Evaluating systemic availability based on ADME (Absorption, Distribution, Metabolism, Excretion) properties 6. Assessing potential health hazards and risks associated with chemical exposure\nWhen providing toxicological evaluations: - Use reliable scientific sources and databases (e.g., PubChem, ECHA, EPA, IARC) - Consider both experimental data and predictive models - Explain your reasoning and cite relevant studies or guidelines - Acknowledge uncertainties and data gaps - Provide a balanced assessment, considering both potential hazards and mitigating factors - Use a weight-of-evidence approach when multiple data sources are available - Classify toxicodynamic activity and systemic availability as high, medium, or low based on  the available evidence and expert judgment - When using read-across, clearly state the basis for the analogy and any limitations\nAdhere to ethical standards in toxicology and maintain scientific objectivity in your assessments.\n          ",
                "tests": 
                [
                    {
                        "provider": {
                            "config": {
                                "temperature": 0
                            },
                            "id": "openai:chat:claude-3-7-sonnet",
                            "label": "Claude 3.7 Sonnet"
                        },
                        "prompt": "Answer the following question with a list of string. List the {effect_type} of {chem_casrn} on human",
                        "vars": {
                            "chem_casrn": "Dioxin or its synonyms with CAS number 1746-01-6",
                            "effect_type": "Toxicokinetic parameters"
                        },
                        "assert": [
                            {
                                "expected_phrases": [
                                    "Half-life: 7\u201311 years"
                                ],
                                "type": "python",
                                "value": "file://scripts/tests.py"
                            }
                        ],
                        "response": {
                            "output": "# Toxicokinetic Parameters of 2,3,7,8-Tetrachlorodibenzo-p-dioxin (TCDD, Dioxin)\n\n**CAS Number: 1746-01-6**\n\n- Half-life: 7-11 years in humans (mean approximately 7.2 years)\n- Volume of distribution: 15-20 L/kg\n- Body fat concentration ratio: 100-400 (highly concentrated in adipose tissue)\n- Oral absorption rate: 50-90% of ingested dose\n- Dermal absorption: Approximately 1-40% depending on vehicle and exposure conditions\n- Plasma protein binding: >97% (primarily bound to lipoproteins)\n- Metabolism: Primarily via CYP1A1, CYP1A2, and CYP1B1\n- Primary metabolites: Hydroxylated derivatives\n- Elimination route: Primarily fecal (>90%), with minor urinary excretion (<10%)\n- Clearance rate: 0.1-0.3 kg of body weight/day\n- Bioaccumulation factor: Very high (estimated at 5,000-10,000)\n- Blood:fat partition coefficient: Approximately 1:100\n- Placental transfer efficiency: Readily crosses placenta\n- Breast milk transfer: High concentration in breast milk (milk:blood ratio of 2:1)\n- Steady-state body burden: 5-10 ng/kg body weight (background exposure)\n- Biological half-life in blood: 5-10 years\n- Apparent volume of distribution at steady state: 1-3 L/kg",
                            "results": {
                                "pass": false,
                                "score": 0,
                                "reason": "All assertions did not pass",
                                "componentResults": [
                                    {
                                        "pass": false,
                                        "reason": "The QUERY asks for the effects of a chemical on humans, while the ANSWER provides toxicokinetic parameters of TCDD, which are not direct effects but rather describe how the chemical is processed in the body. The PHRASE 'Half-life: 7\u201311 years' is a part of the toxicokinetic parameters and not an effect on human health. Therefore, the ANSWER and PHRASE are not relevant to the QUERY."
                                    }
                                ]
                            }
                        }
                    }
                ]
            }
           """)
eval_sets_eval = []
for t in output['tests']:

    model_info = t['provider']
    prompt = t['prompt']
    prompt_info = {'system': output['system_prompt'], 'user': prompt}
    vars_info = t['vars']
    assert_info = t['assert']

    eval_sets_eval.append([assert_info, t['response']['output'], prompt_info['user']])

results = map(eval.getEvaluationResponse, *zip(*eval_sets_eval))

ModuleNotFoundError: No module named 'evaluation'

In [1]:
st = """The positive activity results from chemicals on assays such as microelectrode arrays (MEA), human multi-neurotransmitter receptor (hMNR), and brain organoids can be indicative of potential neurotoxic pathways. These pathways can provide insights into how these chemicals might impact neurological function and development, with implications for conditions like seizures, ADHD, or autism spectrum disorder (ASD). Here are some neurotoxic pathways and their potential relevance:

1. **Ion Channel Modulation:**
   - **Pathway:** Chemicals that modulate ion channels can alter neuronal excitability. MEAs are particularly sensitive to changes in electrical activity across neuronal networks.
   - **Relevance:** Dysregulation of ion channels is linked to seizures due to abnormal neuronal firing. Perturbations in ion channel dynamics can also affect ADHD and ASD through altered neural signaling and connectivity.

2. **Neurotransmitter System Disruption:**
   - **Pathway:** Alterations in neurotransmitter receptors, detectable on hMNR assays, can indicate disruptions in neurotransmission.
   - **Relevance:** Imbalances in neurotransmitter systems (e.g., dopamine, serotonin, GABA) are associated with ADHD and ASD. Overactivation or inhibition can lead to excitotoxicity or dysfunctional neural circuits, influencing seizure risk and developmental disorders.

3. **Oxidative Stress and Mitochondrial Dysfunction:**
   - **Pathway:** Chemicals inducing oxidative stress can impair mitochondrial function, crucial for neuron energy supply.
   - **Relevance:** Oxidative stress and mitochondrial dysfunction contribute to neurodegeneration and neuronal hyperexcitability, underpinning seizure activity and potentially developmental disorders like ASD.

4. **Neuronal Network Formation and Plasticity:**
   - **Pathway:** Disruption in synaptogenesis or synaptic plasticity can be assessed through changes observed in brain organoids.
   - **Relevance:** Proper synaptic connections and plasticity are critical during neurodevelopment. Abnormalities can lead to developmental and functional impairments typical of ADHD and ASD.

5. **Neuroinflammation:**
   - **Pathway:** Inflammatory responses in the brain can be triggered by toxicants, leading to changes in neural activity.
   - **Relevance:** Chronic neuroinflammation is increasingly recognized in neurological disorders, influencing the risk and severity of conditions like seizures, ADHD, and ASD.

These pathways illustrate the complex interplay of neurotoxic effects that can lead to adverse neurological outcomes. Predictive toxicology aims to utilize these insights to identify potential hazards early and understand mechanistic links between chemical exposure and neurodevelopmental/neurological disorders. By integrating data from MEAs, hMNR assays, and brain organoids, researchers can improve risk assessment models and develop more targeted interventions to mitigate these risks."""

print(st.replace('\n', '\\n').replace('\t', '\\t'))

The positive activity results from chemicals on assays such as microelectrode arrays (MEA), human multi-neurotransmitter receptor (hMNR), and brain organoids can be indicative of potential neurotoxic pathways. These pathways can provide insights into how these chemicals might impact neurological function and development, with implications for conditions like seizures, ADHD, or autism spectrum disorder (ASD). Here are some neurotoxic pathways and their potential relevance:\n\n1. **Ion Channel Modulation:**\n   - **Pathway:** Chemicals that modulate ion channels can alter neuronal excitability. MEAs are particularly sensitive to changes in electrical activity across neuronal networks.\n   - **Relevance:** Dysregulation of ion channels is linked to seizures due to abnormal neuronal firing. Perturbations in ion channel dynamics can also affect ADHD and ASD through altered neural signaling and connectivity.\n\n2. **Neurotransmitter System Disruption:**\n   - **Pathway:** Alterations in neur